# Protocol 9 — Validate: can I trust this result?

A CPP **signature** (Protocol 1) and any classifier built on it can look convincing on a tiny dataset and still be an artefact of overfitting, leakage, or chance. This protocol is the **validation checklist**: a short, ordered battery of internal sanity checks that asks *"would this result survive resampling and a shuffled-label control?"* before you attach a biological claim to it.

This protocol is part of the [Protocols catalog (#35)](https://github.com/breimanntools/aaanalysis/issues/35). It follows the standard pipeline-chained structure: *When to use it -> Input -> Run -> Output -> How to interpret -> Common mistakes -> Next step.*

## When to use it

Use this protocol **after** you have a signature and/or a fitted classifier and **before** you report a number or draw a biological conclusion. It answers the reviewer's first question: *"Is this real, or did the model memorise 40 sequences?"*

The biological setting here is the gamma-secretase (`DOM_GSEC`) task: separating **substrate** transmembrane domains (`label=1`) from **non-substrate** TMDs (`label=0`). A trustworthy result means the physicochemical separation we measured is stable under resampling and survives the controls below — not that we fit noise.

The checklist mirrors the validation epic:

| Check | Question it answers |
| --- | --- |
| Repeated stratified CV | Does performance hold across many train/test splits, not one lucky one? |
| Bootstrap CI of the mean | How wide is the uncertainty band around the score? |
| Shuffled-label control | Does the score **collapse** when the labels carry no information? |
| Feature stability | Do the top features keep their effect sizes under resampling? |
| Biological sense | Do the strongest features match known biology? |
| Generalization headroom (learning curve) | Is the task sampling-limited — would more TMDs still raise the score, or has it plateaued? |

All six checks are **internal**: they run on the same 40 TMDs via resampling and in-sample cross-validation. They tell you whether the result is stable and not an artefact; they do **not** replace a true external hold-out (a separate dataset), which is the only way to measure external generalization.

## Input

The same input as Protocol 1: a `df_seq` with one row per protein, a binary `label` column (test = 1, reference = 0), and domain coordinates (`tmd_start` / `tmd_stop`). We rebuild the **signature** (`df_feat`) and the **feature matrix** `X` exactly as in *Protocol 1 - CPP signature*, then validate them.

If you arrived here straight from Protocol 1 you already have `df_feat` and `X`; the cell below reproduces them so this protocol is self-contained. We keep the fixture small (`n=20` -> 40 balanced TMDs) and the cross-validation light so the whole checklist runs in seconds.

In [1]:
import warnings
import numpy as np
import aaanalysis as aa

aa.options["verbose"] = False
aa.options["random_state"] = 42

# Rebuild the Protocol 1 signature on a small balanced fixture
df_seq = aa.load_dataset(name="DOM_GSEC", n=20)      # 20 per class -> 40 rows
labels = np.array(df_seq["label"].to_list())

sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
cpp = aa.CPP(df_parts=df_parts)
df_feat = cpp.run(labels=labels, n_filter=25, n_jobs=1)

# Feature matrix: rows = TMDs, columns = the 25 selected features
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)
X.shape, df_feat.shape

((40, 25), (25, 13))

## Run

Each check is one small cell. We use a `RandomForestClassifier` as a neutral downstream model and **Matthews correlation coefficient (MCC)** as the headline metric: MCC is balanced, ranges from -1 to 1, and reads ~0 for a model that has learned nothing — which makes the shuffled-label control easy to interpret. (AAanalysis ships its own tree classifier, `aa.TreeModel`, whose `.eval` wraps a k-fold CV over feature selections — a feature-set-quality evaluation, per #91; we drive a raw RandomForest here only to keep every control explicit and framework-agnostic.)

### Check 1 - Repeated stratified cross-validation

`RepeatedStratifiedKFold` keeps the 50/50 class balance in every fold and repeats the whole split several times, so the score is an average over many train/test partitions rather than one lucky split.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import matthews_corrcoef, make_scorer

clf = RandomForestClassifier(n_estimators=100, random_state=42)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)  # 15 fits
mcc = make_scorer(matthews_corrcoef)

scores = cross_val_score(clf, X, labels, cv=cv, scoring=mcc)
float(scores.mean()), float(scores.std())

(0.8070022769184196, 0.17911074334561763)

### Check 2 - Bootstrap confidence interval

`comp_bootstrap_ci` resamples the 15 fold scores with replacement (seed-locked) to put a 95% interval around the mean MCC, so we report a band rather than a bare point estimate.

In [3]:
# Check 2 - Bootstrap confidence interval of the mean MCC
# comp_bootstrap_ci resamples the 15 fold scores with replacement (seed-locked).
ci = aa.comp_bootstrap_ci(values=scores, n_rounds=1000, ci=0.95, seed=42)
ci

{'mean': 0.8070022769184196,
 'ci_low': 0.7139891620269944,
 'ci_high': 0.8967358359109805}

### Check 3 - Shuffled-label (negative) control

The single most informative check: permute the labels so they carry no information and confirm the score collapses toward MCC ~ 0. We average over several reshuffles so the null is not one noisy draw.

In [4]:
# Check 3 - Shuffled-label (negative) control
# Permute the labels many times: any structure left is pure chance. A
# trustworthy signature collapses toward MCC ~ 0 here. We average over
# several reshuffles so the null is a mean over many splits, not a single
# noisy point estimate (the same 'many resamples' logic as Check 1).
rng = np.random.default_rng(42)
shuffled_means = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")          # degenerate shuffled folds -> expected MCC warnings
    for _ in range(10):
        labels_shuffled = rng.permutation(labels)
        s = cross_val_score(clf, X, labels_shuffled, cv=cv, scoring=mcc)
        shuffled_means.append(s.mean())
{"real_MCC": round(float(scores.mean()), 3),
 "shuffled_MCC_mean": round(float(np.mean(shuffled_means)), 3),
 "shuffled_MCC_max": round(float(np.max(shuffled_means)), 3)}

{'real_MCC': 0.807, 'shuffled_MCC_mean': 0.051, 'shuffled_MCC_max': 0.252}

### Check 4 - Feature stability under resampling

Recompute each feature's adjusted-AUC effect size on bootstrap resamples and correlate it with the full-data effect sizes. High correlation means the signature reflects real effect sizes, not resampling noise.

In [5]:
# Check 4 - Feature stability under resampling
# Recompute the adjusted AUC (effect size, [-0.5, 0.5]) for every feature on
# bootstrap resamples and correlate it with the full-data effect sizes.
# High correlation = the signature's effect sizes are not resampling artefacts.
# Re-seed here so this cell is reproducible on its own (not dependent on the
# RNG state left by Check 3).
rng = np.random.default_rng(42)
auc_full = aa.comp_auc_adjusted(X=X, labels=labels)
corrs = []
for _ in range(5):
    idx = rng.choice(len(labels), size=len(labels), replace=True)
    while len(np.unique(labels[idx])) < 2:          # keep both classes present
        idx = rng.choice(len(labels), size=len(labels), replace=True)
    auc_b = aa.comp_auc_adjusted(X=X[idx], labels=labels[idx])
    corrs.append(np.corrcoef(auc_full, auc_b)[0, 1])
round(float(np.mean(corrs)), 3)

0.992

### Check 5/6 - Biological sense + generalization headroom (learning curve)

Read the strongest features against known substrate biology, then use a learning curve to ask whether the task is **sampling-limited** — would more TMDs still raise the score, or has the signal plateaued?

For `DOM_GSEC`, the expected substrate physicochemistry around the cleavage site is **helix-destabilizing / helix-breaking propensity, β-turn and flexibility signals** (a locally less rigid, more bendable helix lets gamma-secretase unwind and cleave the TMD). So α-helix, β-turn and folding free-energy subcategories topping the list is the biologically sensible outcome; charge or bulky-hydrophobicity terms dominating instead would be a red flag.

Note: this is **internal** in-sample cross-validation over the same 40 TMDs — it measures sampling headroom, not external generalization. A true external-generalization estimate needs a separate hold-out dataset.

In [6]:
# Check 5/6 - Biological sense + generalization headroom (learning curve)
# Top features by effect size: do the strongest physicochemical properties
# match known substrate biology? For DOM_GSEC we expect helix-destabilizing /
# β-turn / flexibility signals near the cleavage site (read the subcategory).
top = df_feat.reindex(df_feat["abs_auc"].sort_values(ascending=False).index)
display(top[["feature", "category", "subcategory", "mean_dif", "abs_auc"]].head(5))

# Learning curve (INTERNAL in-sample CV, no external hold-out): does the score
# still climb with more TMDs (the task is sampling-limited, more data would
# help) or has it plateaued (the signal is captured)? We start at 0.6 of the
# training fold so the smallest size keeps both classes (a smaller size makes
# MCC undefined, the tiny-fold trap flagged under Common mistakes).
from sklearn.model_selection import learning_curve
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sizes, train_sc, test_sc = learning_curve(
        clf, X, labels, cv=cv, scoring=mcc,
        train_sizes=np.linspace(0.6, 1.0, 4))
{"train_sizes": sizes.tolist(),
 "test_MCC_by_size": np.round(test_sc.mean(axis=1), 3).tolist()}

,feature,category,subcategory,mean_dif,abs_auc
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,0.299,0.444
1,"TMD_C_JMD_C-Pattern(N,4,8)-BEGF750101",Conformation,α-helix,0.299,0.444
2,"TMD_C_JMD_C-Pattern(N,4,8,12)-CRAJ730103",Conformation,β-turn,-0.251,0.431
3,"TMD_C_JMD_C-Pattern(N,4,8,12)-MUNV940102",Energy,Free energy (folding),-0.148,0.422
4,"TMD_C_JMD_C-PeriodicPattern(C,i+4/4,2)-BEGF750103",Conformation,β-turn,-0.139,0.409


{'train_sizes': [19, 23, 27, 32],
 'test_MCC_by_size': [0.696, 0.778, 0.792, 0.807]}

## Output

A small panel of numbers, one per check — nothing is plotted, everything is a scalar you can paste into a methods section:

- `scores.mean()` / `scores.std()` - mean repeated-CV MCC and its spread across 15 fits.
- `ci` - dictionary `{'mean', 'ci_low', 'ci_high'}`: the 95% bootstrap interval of the mean MCC. Caveat: the 15 `RepeatedStratifiedKFold` folds share most of their training data, so they are not independent samples; bootstrapping them understates uncertainty, making this band a *lower bound* on the true interval. The only honest test of generalization is an external hold-out (a separate dataset); for a tighter internal estimate, bootstrap an independent per-protein score (e.g. leave-one-protein-out correctness). Note `comp_per_protein_ap` is *not* applicable here: it is a residue-level / windowed site-prediction metric (per-residue score vectors + positive-site positions), undefined for this `DOM_GSEC` domain-level task with one TMD and a single binary label per protein.
- `real_MCC` vs `shuffled_MCC_mean` / `shuffled_MCC_max` - the headline score against its negative control, averaged (and worst-cased) over 10 independent label permutations.
- the stability correlation - mean Pearson correlation (linear agreement) of per-feature effect sizes across bootstrap resamples (1.0 = an identical effect-size profile every time).
- `top` - the five strongest features for the biological-sense eyeball check.
- `test_MCC_by_size` - MCC as a function of training-set size (the learning curve).

## How to interpret

| Output | Trustworthy result | Warning sign |
| --- | --- | --- |
| repeated-CV `MCC` | high and stable (low `std`) | high mean but large `std` -> split-dependent |
| bootstrap `ci` | narrow, well above 0 | wide band crossing ~0 -> under-powered |
| `shuffled_MCC_mean` | **collapses** far below the real MCC (toward 0) | stays high -> leakage / the metric is not measuring signal |
| stability correlation | close to 1.0 | low / variable -> effect sizes are resampling noise |
| `top` features | subcategories match expected GSEC-substrate biology (helix-destabilizing / β-turn / flexibility near the cleavage site) | top features are biologically implausible (e.g. charge / bulk dominating) -> suspect overfitting |
| learning curve (internal CV) | rising then plateauing | flat and low -> features carry little signal; still steeply rising -> task is sampling-limited, more data needed |

The single most informative line is **real vs shuffled**: if scrambling the labels does *not* destroy performance, the pipeline is leaking information and no other number can be trusted. On this fixture the real MCC sits high while the shuffled control drops sharply toward 0 - the separation tracks the labels, not an artefact.

Caveat: with only 40 TMDs every number is *under-powered*, and all six checks are internal (resampling / in-sample CV on the same TMDs). Treat this protocol as a gate (does it survive the controls?), not as a final or external performance estimate — report the bootstrap interval, never a bare point estimate, and confirm external generalization on a separate dataset.

## Common mistakes

- **No shuffled-label control.** A high CV score alone proves nothing; only the *gap* between real and shuffled performance does. Always run the negative control.
- **Plain K-fold instead of stratified.** With 40 rows an unstratified split can leave a fold with one class, making MCC undefined or wildly unstable. Use `RepeatedStratifiedKFold`.
- **Reporting a point estimate.** A single MCC hides the uncertainty. Report the bootstrap CI (`comp_bootstrap_ci`) of the mean.
- **Re-selecting features inside the loop is skipped here for speed** - on a real study, run CPP feature selection *inside* each CV fold to avoid selection leakage; reusing one global `df_feat` mildly optimistically biases the score.
- **Over-reading a single feature.** Interpret the *signature* as blocks of related features; check effect-size stability before any biological claim.
- **Using `len(df_seq)` for class sizes.** `load_dataset(name=..., n=N)` returns `2N` rows (N per class); use the `label` column.

## Next step

You have run the internal validation checklist - repeated stratified CV with a bootstrap CI, a shuffled-label control, feature-stability under resampling, a biological-sense read, and a learning curve (generalization headroom). If the result survived all of them, it is ready to report — with the caveat that external generalization still needs a separate hold-out dataset.

- **Back to the start** - revisit the signature with confidence: *Protocol 1 - CPP signature*.
- **Pick the right validation per task level** - the [cheat-sheet decision tree (#81)](https://github.com/breimanntools/aaanalysis/issues/81) maps residue / domain / protein tasks to the checks that matter most.
- **Quantify uncertainty per protein** - for *residue-level / windowed site-prediction* tasks (the residue-level protocol, not this domain task), pair `comp_per_protein_ap` with `comp_bootstrap_ci` for protein-level confidence intervals.